# CAM equilibrium check

This notebook is for **CAM monthly history files**.

*Author: ChatGPT (22 April 2026)*

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob

import sys
from pathlib import Path

from boreal_forest_expansion.core import open_mfdataset_selected, add_derived_cam_fields, reduce_series, annual_mean
from boreal_forest_expansion.diagnostics import equilibrium_test, moving_window_trends

In [2]:
path_to_archive='/nird/datapeak/NS9188K/adelez/BRL-FRST-XPSN_archive/NF2000norbc_tropstratchem_spinup_f19_f19/atm/hist/'

In [3]:
CAM_GLOB = path_to_archive+"*.cam.h0.*.nc"
CAM_VARS = ["RESTOM","TREFHT","PRECT","FSNT","FLNT","PSL","CLDTOT"]

In [4]:
files = sorted(glob.glob(CAM_GLOB))

print(f"Number of files: {len(files)}")
print(f"First file: {files[0]}")

ds0 = xr.open_dataset(files[0])
ds0

Number of files: 240
First file: /nird/datapeak/NS9188K/adelez/BRL-FRST-XPSN_archive/NF2000norbc_tropstratchem_spinup_f19_f19/atm/hist/NF2000norbc_tropstratchem_spinup_f19_f19.cam.h0.0001-01.nc


<xarray.Dataset> Size: 1GB
Dimensions:               (lat: 96, zlon: 1, nbnd: 2, lon: 144, lev: 32,
                           ilev: 33, cosp_prs: 7, cosp_tau: 7, cosp_scol: 10,
                           cosp_ht: 40, cosp_sr: 15, cosp_sza: 5,
                           cosp_dbze: 15, cosp_htmisr: 16, cosp_tau_modis: 7,
                           cosp_reffice: 6, cosp_reffliq: 6, time: 1)
Coordinates: (12/17)
  * lat                   (lat) float64 768B -90.0 -88.11 -86.21 ... 88.11 90.0
  * zlon                  (zlon) float64 8B 0.0
  * lon                   (lon) float64 1kB 0.0 2.5 5.0 ... 352.5 355.0 357.5
  * lev                   (lev) float64 256B 3.643 7.595 14.36 ... 976.3 992.6
  * ilev                  (ilev) float64 264B 2.255 5.032 10.16 ... 985.1 1e+03
  * cosp_prs              (cosp_prs) float64 56B 9e+04 7.4e+04 ... 9e+03
    ...                    ...
  * cosp_dbze             (cosp_dbze) float64 120B -72.5 -42.5 ... 17.5 50.0
  * cosp_htmisr           (cosp_htmisr) float64 128B 0.0 250.0 ... 1.8e+04
  * cosp_tau_modis        (cosp_tau_modis) float64 56B 0.15 0.8 ... 41.5 100.0
  * cosp_reffice          (cosp_reffice) float64 48B 5e-06 1.5e-05 ... 7.5e-05
  * cosp_reffliq          (cosp_reffliq) float64 48B 4e-06 9e-06 ... 2.5e-05
  * time                  (time) object 8B 0001-02-01 00:00:00
Dimensions without coordinates: nbnd
Data variables: (12/2546)
    zlon_bnds             (zlon, nbnd) float64 16B ...
    gw                    (lat) float64 768B ...
    hyam                  (lev) float64 256B ...
    hybm                  (lev) float64 256B ...
    P0                    float64 8B ...
    hyai                  (ilev) float64 264B ...
    ...                    ...
    wet_OM                (time, lat, lon) float32 55kB ...
    wet_SALT              (time, lat, lon) float32 55kB ...
    wet_SO2               (time, lat, lon) float32 55kB ...
    wet_SO2_S             (time, lat, lon) float32 55kB ...
    wet_SULFATE           (time, lat, lon) float32 55kB ...
    wet_SULFATE_S         (time, lat, lon) float32 55kB ...
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              NF2000norbc_tropstratchem_spinup_f19_f19
    logname:           adelez
    host:              
    initial_file:      NHIST_tropstratchem_01_f19_tn14_r1990_s01_20241118.cam...
    topography_file:   /cluster/shared/noresm/inputdata/atm/cam/topo/fv_1.9x2...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  month_1

## 4. Open CAM dataset

In [7]:
cam_ds = open_mfdataset_selected(CAM_GLOB, CAM_VARS)
cam_ds = add_derived_cam_fields(cam_ds)

print("Time span:", str(cam_ds.time.values[0]), "to", str(cam_ds.time.values[-1]))
print(cam_ds)

Keeping variables: ['TREFHT', 'PRECT', 'FSNT', 'FLNT', 'PSL', 'CLDTOT', 'time', 'time_bnds']
Dropping 2555 variables
Time span: 0001-01-15 00:00:00 to 0020-12-15 00:00:00
<xarray.Dataset> Size: 93MB
Dimensions:    (time: 240, nbnd: 2, lat: 96, lon: 144)
Coordinates:
  * time       (time) object 2kB 0001-01-15 00:00:00 ... 0020-12-15 00:00:00
Dimensions without coordinates: nbnd, lat, lon
Data variables:
    time_bnds  (time, nbnd) object 4kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    CLDTOT     (time, lat, lon) float32 13MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    FLNT       (time, lat, lon) float32 13MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    FSNT       (time, lat, lon) float32 13MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    PRECT      (time, lat, lon) float32 13MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    PSL        (time, lat, lon) float32 13MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    TREFHT     (time, l

In [1]:
cam_ds["time"].encoding["units"] = "days since 0001-01-01"
cam_ds["time"].encoding["calendar"] = "noleap"  # or "gregorian", depending on your data

cam_ds.to_netcdf("cam_spinup_dataset.nc")

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/cluster/home/adelez/pyenv/lib64/python3.9/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/cluster/home/adelez/pyenv/lib64/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/cluster/home/adelez/pyenv/lib64/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/cluster/home/adelez/pyenv/lib64/python3.9/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should encode the Optional
  File "/cluster/home/adelez/pyenv/lib64/python3.9/site-packages/traitlets/traitlets.py", line 649, in get
    value = self._validate(obj, default

NameError: name 'cam_ds' is not defined

In [ ]:
OUTDIR = "./cam_equilibrium_output"

# Equilibrium parameters
LAST_N_YEARS = 10
MOVING_WINDOW = 10
ROLLING_WINDOW = 5
REL_DRIFT_THRESHOLD = 0.01
PVAL_THRESHOLD = 0.05

REGIONS = {
    "global": None,
    "tropics": (-23.5, 23.5, -180.0, 180.0),
    "NH_extratropics": (23.5, 90.0, -180.0, 180.0),
    "SH_extratropics": (-90.0, -23.5, -180.0, 180.0),
    "amazon": (-20.0, 10.0, -80.0, -45.0),
    "europe": (35.0, 72.0, -10.0, 40.0),
    "boreal": (50.0, 75.0, -180.0, 180.0),
    "arctic": (66.5, 90.0, -180.0, 180.0),
}

Path(OUTDIR).mkdir(parents=True, exist_ok=True)

## 5. Check available variables

In [10]:
cam_available = [v for v in CAM_VARS if v in cam_ds.data_vars]
cam_missing = [v for v in CAM_VARS if v not in cam_ds.data_vars]
print("CAM available:", cam_available)
print("CAM missing:", cam_missing)


NameError: name 'cam_ds' is not defined

## 6. Plot one CAM variable interactively

In [ ]:
def plot_one_cam_variable(variable: str, region_name: str = "global"):
    region_bounds = REGIONS[region_name]
    da = maybe_convert_units(cam_ds[variable], variable)
    series = reduce_series(da, region_bounds)
    ann = annual_mean(series)
    summary, y_last, fitted_last = equilibrium_test(
        ann, last_n_years=LAST_N_YEARS,
        rel_drift_threshold=REL_DRIFT_THRESHOLD,
        pval_threshold=PVAL_THRESHOLD,
    )
    summary.variable = variable
    summary.region = region_name

    years = pd.to_datetime(ann["time"].values).year.astype(float)
    vals = ann.values.astype(float)
    roll = rolling_mean(vals, ROLLING_WINDOW)
    units = get_units(ann)

    plt.figure(figsize=(9, 5.5))
    plt.plot(years, vals, marker="o", linewidth=1.4, label="Annual mean")
    plt.plot(years, roll, linewidth=2.0, label=f"{ROLLING_WINDOW}-yr rolling mean")
    plt.plot(y_last, fitted_last, linestyle="--", linewidth=2.0, label=f"Late-run trend ({LAST_N_YEARS} yr)")
    plt.title(f"CAM | {variable} | {region_name}\nrel. change last {LAST_N_YEARS} yr = {summary.rel_change_last_period:.3%}, slope = {summary.slope_per_decade:.3g} / decade, eq = {summary.equilibrium_flag}")
    plt.xlabel("Year")
    plt.ylabel(f"{variable} ({units})" if units else variable)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    trend_df = moving_window_trends(ann, MOVING_WINDOW)
    if not trend_df.empty:
        plt.figure(figsize=(9, 4.8))
        plt.plot(trend_df["year_end"], trend_df["slope_per_decade"], marker="o", linewidth=1.5)
        plt.axhline(0.0, linestyle="--", linewidth=1.0)
        plt.title(f"CAM | {variable} | {region_name}\nMoving {MOVING_WINDOW}-year trend")
        plt.xlabel("End year of moving window")
        plt.ylabel(f"Slope per decade [{units}/decade]" if units else "Slope per decade")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    return summary, ann, trend_df


## 7. Example CAM plots

In [ ]:
summary_restom, ann_restom, trend_restom = plot_one_cam_variable("RESTOM", "global")
summary_trefht, ann_trefht, trend_trefht = plot_one_cam_variable("TREFHT", "global")
summary_restom, summary_trefht
